In [1]:
%load_ext cuml.accel
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from cuml.cluster import DBSCAN
from sklearn.metrics import classification_report, roc_auc_score

[2025-06-14 10:14:53.156] [CUML] [info] cuML: Installed accelerator for sklearn.
[2025-06-14 10:15:18.810] [CUML] [info] cuML: Installed accelerator for umap.
[2025-06-14 10:15:18.908] [CUML] [info] cuML: Installed accelerator for hdbscan.
[2025-06-14 10:15:18.908] [CUML] [info] cuML: Successfully initialized accelerator.


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kartik2112/fraud-detection")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fraud-detection


In [3]:
df_train = pd.read_csv(path + "/fraudTrain.csv")
df_test = pd.read_csv(path + "/fraudTest.csv")

In [15]:
df_train['is_fraud'].value_counts()

,count
is_fraud,
0,1289169
1,7506


In [4]:
numerical_cols = ['amt', 'city_pop', 'unix_time', 'distance', 'hour', 'day_of_week', 'month']
categorical_cols = ['category']

In [5]:
def add_features(df):
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month'] = df['trans_date_trans_time'].dt.month

    def haversine(lat1, lon1, lat2, lon2):
        R = 6371
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return R * c

    df['distance'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
    return df

df_train = add_features(df_train)
df_test = add_features(df_test)


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ])

In [10]:
X_train = df_train[numerical_cols + categorical_cols]
X_train_preprocessed = preprocessor.fit_transform(X_train)

In [11]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
df_train['cluster'] = dbscan.fit_predict(X_train_preprocessed)
df_train['is_anomaly'] = df_train['cluster'] == -1

[2025-06-14 10:57:49.782] [CUML] [warning] Batch size limited by the chosen integer type (4 bytes). 1920 -> 1656. Using the larger integer type might result in better performance


In [12]:
X_test = df_test[numerical_cols + categorical_cols]
X_test_preprocessed = preprocessor.transform(X_test)

df_test['cluster'] = dbscan.fit_predict(X_test_preprocessed)
df_test['is_anomaly'] = df_test['cluster'] == -1

[2025-06-14 11:30:08.978] [CUML] [warning] Batch size limited by the chosen integer type (4 bytes). 4523 -> 3864. Using the larger integer type might result in better performance


In [13]:
print("Sample of test set clustering results:")
print(df_test[['amt', 'hour', 'category', 'cluster', 'is_anomaly', 'is_fraud']].head(10))
print("\nNumber of anomalies detected in test set:", df_test['is_anomaly'].sum())
print("Number of actual fraud cases in test set:", df_test['is_fraud'].sum())

Sample of test set clustering results:
      amt  hour        category  cluster  is_anomaly  is_fraud
0    2.86    12   personal_care       -1        True         0
1   29.84    12   personal_care        0       False         0
2   41.28    12  health_fitness        1       False         0
3   60.05    12        misc_pos        2       False         0
4    3.19    12          travel        3       False         0
5   19.55    12       kids_pets        4       False         0
6  133.93    12  health_fitness       -1        True         0
7   10.37    12   personal_care        0       False         0
8    4.37    12    shopping_pos        5       False         0
9   66.54    12     food_dining        6       False         0

Number of anomalies detected in test set: 50706
Number of actual fraud cases in test set: 2145


In [14]:
print("\nClassification Report (Anomaly as Fraud Prediction on Test Set):")
print(classification_report(df_test['is_fraud'], df_test['is_anomaly'], target_names=['Not Fraud', 'Fraud']))
print("ROC-AUC Score on Test Set:", roc_auc_score(df_test['is_fraud'], df_test['is_anomaly']))


Classification Report (Anomaly as Fraud Prediction on Test Set):
              precision    recall  f1-score   support

   Not Fraud       1.00      0.91      0.95    553574
       Fraud       0.03      0.60      0.05      2145

    accuracy                           0.91    555719
   macro avg       0.51      0.76      0.50    555719
weighted avg       0.99      0.91      0.95    555719

ROC-AUC Score on Test Set: 0.7551296877591104


In [17]:
import joblib
joblib.dump(dbscan, 'dbscan_fraud_model.pkl')

['dbscan_fraud_model.pkl']